# Feature Engineering for Time Series Regression

In this notebook, we explore **feature engineering techniques** specifically designed for time series regression tasks, using electricity load demand data as our example. Feature engineering is the process of creating new input features from raw data to help machine learning models capture important patterns and improve forecasting accuracy.


For time series data, this often means creating features that represent temporal dependencies, such as lagged values, rolling statistics, and seasonal indicators. These features allow models to learn from the past and recognize repeating patterns, trends, and seasonality in the data.


Throughout this notebook, we will:


- Explain the intuition and mathematics behind key time series features (starting with lags),
- Show how to implement these features in Python using the `polars` and `mlforecast` libraries,
- Discuss best practices for using these features in forecasting models.


By the end, you'll have a solid understanding of how to engineer features that make your time series regression models more powerful and interpretable.

In [ ]:
import polars as pl
import plotly.express as px
import seaborn as sns
from utilsforecast.plotting import plot_series
from statsforecast import StatsForecast
from mlforecast import MLForecast
from mlforecast.lag_transforms import RollingMean
from utilsforecast.losses import *
from utilsforecast.evaluation import evaluate
from mlforecast import MLForecast
from plotting_utils import plotly_series as plot_series, plot_acf
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

import plotly.graph_objects as go


In [ ]:
data = pl.read_parquet(
    "data/london_smart_meters/preprocessed/london_smart_meters_merged_block_0-7.parquet"
)
timestamp = data.group_by("LCLid").agg(
    pl.datetime_range(
        start=pl.col("start_timestamp"),
        end=pl.col("start_timestamp").dt.offset_by(
            pl.format("{}m", pl.col("series_length").sub(1).mul(30))
        ),
        interval="30m",
    ).alias("ds"),
)
data = timestamp.join(data, on="LCLid", how="inner").rename(
    {"LCLid": "unique_id", "energy_consumption": "y"}
)
data.head(5)

In [ ]:
id_ = "unique_id"
time_ = "ds"
target_ = "y"
id_col = pl.col(id_)
time_col = pl.col(time_)
target_col = pl.col(target_)

In [ ]:
data = (
    data.filter(pl.col("file").eq("block_7"))
    .select(
        [
            time_,
            id_,
            target_col,
            # "Acorn",
            # "Acorn_grouped",
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
    .explode(
        [
            time_,
            target_,
            # "holidays",
            # "visibility",
            # "windBearing",
            # "temperature",
            # "dewPoint",
            # "pressure",
            # "apparentTemperature",
            # "windSpeed",
            # "precipType",
            # "icon",
            # "humidity",
            # "summary",
        ]
    )
    .with_columns(target_col.forward_fill().backward_fill())
)
data.head()

In [ ]:
selected_id = "MAC000193"
data = data.filter(id_col.eq(selected_id)).with_columns(
    target_col.forward_fill().backward_fill()
)
data.head()


# Feature Engineering: Lag Features in Time Series Forecasting

In time series forecasting, especially for electricity load demand, **lag features** are among the most important tools for capturing temporal dependencies.

## What is a Lag?
A **lag** is simply a previous value of the time series. For example, the value at time $t-1$ (yesterday, or the previous half-hour) is called the lag-1 value.

Mathematically, for a time series $y_t$, the lag-$k$ value is $y_{t-k}$.

## Why Use Lags?
- **Autocorrelation:** Electricity demand at a given time is often similar to recent past values.
- **Seasonality:** Lags like 48 (same time yesterday for half-hourly data) or 336 (same time last week) help capture daily and weekly cycles.
- **Model Memory:** Lags give your model the 'memory' to learn from past behavior.

## How to Add Lags with MLForecast
The [MLForecast](https://nixtla.github.io/mlforecast/) library makes it easy to add lag features for machine learning models.

Below, we add lag features for 1, 2, 48, and 336 steps (previous half-hour, previous hour, previous day, previous week) to our electricity demand data.
code
python


In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lags=[1, 2, 48, 336],
    freq="30m",
)

mlf.preprocess(data).head(2)

## How Many Lags Should You Use for Half-Hourly Electricity Load Data?

Choosing the right number and type of lag features is crucial for building effective time series forecasting models. For half-hourly electricity load data, this decision is guided by both **domain knowledge** and **data-driven analysis**.

---

### 1. **Understanding the Data Frequency**

- **Half-hourly data** means each observation represents a 30-minute interval.
- There are **48 intervals per day** ($24 \text{ hours} \times 2 = 48$).
- There are **336 intervals per week** ($48 \text{ intervals/day} \times 7 \text{ days} = 336$).

---

### 2. **Key Lags to Consider**

#### **Short-Term Lags**
- **Lag 1** ($y_{t-1}$): The previous half-hour. Captures immediate autocorrelation.
- **Lag 2** ($y_{t-2}$): One hour ago. Useful for short-term persistence.

#### **Daily Seasonality**
- **Lag 48** ($y_{t-48}$): Same time yesterday. Captures daily cycles and routines.

#### **Weekly Seasonality**
- **Lag 336** ($y_{t-336}$): Same time last week. Captures weekly patterns (e.g., weekdays vs. weekends).

---

### 3. **Why These Lags?**

- **Electricity demand** is highly repetitive:
    - People tend to follow similar routines each day and week.
    - Weather, work schedules, and social patterns drive strong daily and weekly cycles.
- **Including these lags** helps the model "remember" what happened at similar times in the past, which is often predictive of the future.

---

### 4. **Should You Add More Lags?**

- **Additional lags** (e.g., lag 3, lag 4, lag 24, lag 72, etc.) can sometimes help, especially if:
    - There are **complex autocorrelation structures** (e.g., effects from 1.5 days ago).
    - You want to capture **longer-term dependencies**.
- **But:** Adding too many lags can:
    - Increase model complexity and risk of overfitting.
    - Slow down training and prediction.

---

### 5. **How to Decide?**

- **Start simple:** Use the most interpretable and relevant lags (1, 2, 48, 336).
- **Visualize autocorrelation:** Plot the autocorrelation function (ACF) to see which lags are most informative.
- **Experiment:** Try adding/removing lags and evaluate model performance using cross-validation.

---

### 6. **Summary Table**

| Lag Value | Meaning                  | Formula         |
|-----------|--------------------------|-----------------|
| 1         | Previous half-hour       | $y_{t-1}$       |
| 2         | One hour ago             | $y_{t-2}$       |
| 48        | Same time yesterday      | $y_{t-48}$      |
| 336       | Same time last week      | $y_{t-336}$     |

---

### 7. **Key Takeaway**

> For half-hourly electricity load forecasting, **lags of 1, 2, 48, and 336** are a strong starting point, as they capture both short-term memory and key seasonal patterns. You can always refine your lag selection based on data exploration and model validation.

---

**Next:** We'll visualize the autocorrelation in our data to see these patterns in action!

## Visualizing Autocorrelation: The ACF Plot

To understand which past values (lags) are most predictive of future electricity demand, we use the **Autocorrelation Function (ACF)**. The ACF shows how strongly the time series is correlated with its own past values at different lags.

- **High autocorrelation at lag $k$** means that the value $y_{t-k}$ is similar to $y_t$.
- **Peaks at lags 48 and 336** would indicate strong daily and weekly seasonality.

Let's plot the ACF for our selected household (`MAC000193`) using Plotly for interactivity.


**What to look for:**  
- Notice the spikes at lags 48 and 336, which correspond to daily and weekly cycles.
- This visualization helps confirm why we chose these lags for feature engineering.

---

**Tip:**  
If you see strong autocorrelation at other lags, consider adding them as features! Always validate their usefulness with cross-validation or out-of-sample testing.

In [ ]:
# Extract the target series as a numpy array for ACF calculation
y = data.get_column(target_).drop_nulls().to_numpy()

# Plot the autocorrelation function up to lag 336 (one week)
fig = plot_acf(
    y, nlags=336, title="Autocorrelation Function (ACF) of Electricity Demand"
)
fig.show()

fig = plot_acf(
    np.diff(y),
    nlags=336,
    title="Autocorrelation Function (ACF) of Diff Electricity Demand",
)
fig.show()

### Interpreting the ACF Plot: Why Lags 1, 2, 48, and 336 Make Sense

Looking at the Autocorrelation Function (ACF) plot above, we can see clear spikes at several key lags:

- **Lag 1 and Lag 2:**  
    There is strong autocorrelation at lag 1 ($y_{t-1}$) and lag 2 ($y_{t-2}$), indicating that the electricity demand in the most recent half-hour and one hour ago are highly predictive of the current value. This makes sense, as electricity usage tends to change gradually rather than abruptly.

- **Lag 48:**  
    A prominent spike at lag 48 ($y_{t-48}$) reflects daily seasonality. Since our data is half-hourly, lag 48 corresponds to the same time on the previous day. This suggests that people's routines and daily cycles have a strong influence on electricity demand.

- **Lag 336:**  
    Another clear peak at lag 336 ($y_{t-336}$) highlights weekly seasonality. This means that the demand at the same time last week is also a good predictor, likely due to repeating weekly patterns (such as workdays vs. weekends).

**In summary:**  
The ACF plot visually confirms that including lags 1, 2, 48, and 336 as features is a sound choice for our forecasting model. These lags capture both short-term memory and the main seasonal cycles present in the data, providing our model with the most relevant historical context for accurate predictions.

# Rolling Window Aggregation in Time Series

Rolling window aggregation is a powerful technique in time series analysis. It involves calculating summary statistics (like mean, sum, min, max, etc.) over a moving window of past observations. This helps capture local trends, smooth out noise, and provide context for each time point.

---

## What is a Rolling Window?

A **rolling window** of size $k$ at time $t$ includes the values $\{y_{t-k+1}, y_{t-k+2}, \ldots, y_t\}$. For each time step, we slide this window forward and compute an aggregation.

---

## Common Rolling Aggregation Methods

Here are the most frequently used rolling window aggregations:

| Method         | Description                                               | Formula (window size $k$)         |
|----------------|----------------------------------------------------------|------------------------------------|
| **Mean**       | Average of values in the window                          | $\displaystyle \frac{1}{k} \sum_{i=0}^{k-1} y_{t-i}$ |
| **Sum**        | Total sum of values in the window                        | $\displaystyle \sum_{i=0}^{k-1} y_{t-i}$             |
| **Min**        | Minimum value in the window                              | $\min\{y_{t-k+1}, \ldots, y_t\}$  |
| **Max**        | Maximum value in the window                              | $\max\{y_{t-k+1}, \ldots, y_t\}$  |
| **Std**        | Standard deviation of values in the window               | $\sqrt{\frac{1}{k} \sum_{i=0}^{k-1} (y_{t-i} - \bar{y})^2}$ |
| **Median**     | Middle value in the sorted window                        | $\text{median}\{y_{t-k+1}, \ldots, y_t\}$ |
| **Quantile**   | Value at a specified percentile in the window            | e.g., 90th percentile             |
| **Skewness**   | Measure of asymmetry in the window                       |                                    |
| **Kurtosis**   | Measure of "tailedness" in the window                    |                                    |

---

## Why Use Rolling Aggregations?

- **Smoothing:** Rolling means or medians help reduce noise and reveal underlying trends.
- **Feature Engineering:** Rolling statistics provide the model with information about recent behavior (e.g., "Is demand unusually high compared to the recent average?").
- **Anomaly Detection:** Large deviations from rolling statistics can signal outliers or unusual events.

---

## Practical Example

Suppose you want to know the average electricity demand over the past 24 hours (48 half-hour intervals) at each time point. You would use a rolling mean with a window size of 48.

---

## Next Steps

We'll now demonstrate how to compute rolling window features using the `mlforecast` and `polars` libraries, and discuss best practices for choosing window sizes and aggregation types.

In [ ]:
from mlforecast.lag_transforms import RollingMean, RollingStd

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={
        1: [
            RollingMean(window_size=3),
            RollingMean(window_size=6),
            RollingMean(window_size=12),
            RollingMean(window_size=48),
            RollingStd(window_size=3),
            RollingStd(window_size=6),
            RollingStd(window_size=12),
            RollingStd(window_size=48),
        ],
    },
    freq="30m",
)

mlf.preprocess(data).head(10)

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={
        48: [
            RollingMean(window_size=7),
            RollingMean(window_size=14),
            RollingStd(window_size=7),
            RollingStd(window_size=14),
        ]
    },
    freq="30m",
)

mlf.preprocess(data).head(10)

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={
        336: [
            RollingMean(window_size=4),
            RollingMean(window_size=8),
            RollingStd(window_size=4),
            RollingStd(window_size=8),
        ]
    },
    freq="30m",
)

mlf.preprocess(data).head(10)

## Why We Never Use "Lag 0" in Time Series Feature Engineering

When creating lag features or rolling window statistics for time series forecasting, you might wonder:  
**Why do we always start with lag 1, and never use lag 0?**

Let's break this down step by step:

---

### 1. **What Does "Lag 0" Mean?**

- **Lag 0** refers to the value of the target variable at the current time step, $y_t$.
- **Lag 1** is the value at the previous time step, $y_{t-1}$.
- **Lag 2** is two steps back, $y_{t-2}$, and so on.

---

### 2. **Why Not Use Lag 0? (The Data Leakage Problem)**

- In forecasting, our goal is to predict the value at time $t$ ($y_t$) using only information available **up to time $t-1$**.
- If we include $y_t$ (lag 0) as a feature, we're giving the model access to the value it's supposed to predict!
- This is called **data leakage**: the model "cheats" by seeing the answer during training, leading to unrealistically good results that won't generalize to new data.

**Example:**  
Suppose you're forecasting tomorrow's electricity demand. If you let the model see tomorrow's actual demand as an input, it will always be perfect—but this is impossible in real life.

---

### 3. **How Nixtla Libraries Prevent Data Leakage**

- Libraries like **Nixtla's MLForecast** are designed to **enforce this rule**.
- When you specify lags (e.g., `[1, 2, 48, 336]`), the library will only use past values—never the current or future value.
- If you try to use lag 0, Nixtla will raise an error or ignore it, protecting you from accidental leakage.

---

### 4. **Rolling Windows: Why the Window Stops at $t-1$**

When you compute a rolling statistic (like a rolling mean) with lag 1, here's what happens:

- **At time $t$**, the rolling mean with lag 1 and window size $k$ is calculated using the values from $y_{t-k}$ up to $y_{t-1}$.
- **It never includes $y_t$ itself.**

**Mathematically:**  
For a rolling mean of window size $k$ at time $t$:
$$
\text{RollingMean}_{t} = \frac{1}{k} \sum_{i=1}^{k} y_{t-i}
$$

This ensures that **only past information** is used, just like in a real forecasting scenario.

---

### 5. **Summary Table**

| Lag Value | What It Refers To | Used for Feature Engineering? | Why/Why Not?                |
|-----------|-------------------|------------------------------|-----------------------------|
| 0         | $y_t$ (current)   | ❌ Never                     | Would cause data leakage    |
| 1         | $y_{t-1}$         | ✅ Yes                       | Only uses past information  |
| 2         | $y_{t-2}$         | ✅ Yes                       | Only uses past information  |

---

### 6. **Key Takeaway**

> **Always use lag 1 or greater for time series features.  
> Never use lag 0, as it would leak the answer to your model.  
> Nixtla's libraries enforce this best practice to keep your models honest and reliable.**

---

## Choosing Lag Transformations and Their Windows in Time Series Feature Engineering

When engineering features for time series forecasting, **lag transformations** allow us to summarize the recent history of the target variable. In Nixtla's `mlforecast`, you can specify *how far back* to look (the lag) and *what statistic* to compute (mean, std, sum, etc.).

### How Lag Transformations Work in Nixtla

- The **key** in the lag transformation dictionary (e.g., `1`, `7`, `48`, `336`) tells the library **how far back** to look.
    - For example, `1` means the statistic is computed using values up to $t-1$ (the most recent available value before the prediction time).
    - `7` means the statistic is computed using values up to $t-7$ (seven steps back from the current time).
- The **value** is a list of transformation objects (e.g., `RollingMean`, `RollingStd`), each with its own window size.

**Example:**
```python
lag_transforms = {
    1: [RollingMean(window_size=3)],   # Mean of y_{t-3}, y_{t-2}, y_{t-1}
    7: [RollingStd(window_size=7)],    # Std of y_{t-13} to y_{t-7}
}
```

---

### Which Lag Transformations Should You Use?

#### 1. **Short-Term Memory (Recent Lags)**
- **Why:** Captures immediate trends and short-term fluctuations.
- **How:** Use small lags (e.g., `1`, `2`, `3`) with short window sizes (e.g., 3, 6, 12).
- **Example:** Rolling mean of the last 3 half-hours up to $t-1$.

#### 2. **Daily Patterns**
- **Why:** Electricity demand often repeats daily.
- **How:** Use lag `48` (for half-hourly data: 48 steps = 1 day) with window sizes that cover several days.
- **Example:** Rolling mean of the same time over the past 7 days (window size 7, lag 48).

#### 3. **Weekly Patterns**
- **Why:** Weekly routines (weekdays vs. weekends) are strong predictors.
- **How:** Use lag `336` (48 steps × 7 days) with window sizes that cover multiple weeks.
- **Example:** Rolling mean of the same time over the past 4 weeks (window size 4, lag 336).

---

### Best Practices for Choosing Lags and Windows

| Lag Key | What It Captures         | Typical Window Size | Example Feature Description                |
|---------|-------------------------|--------------------|--------------------------------------------|
| 1       | Short-term memory       | 3, 6, 12           | Mean of last 3 values up to $t-1$          |
| 48      | Daily seasonality       | 7, 14              | Mean of same time over past 7 days         |
| 336     | Weekly seasonality      | 4, 8               | Mean of same time over past 4 weeks        |

- **Short lags** (1, 2, 3): Use for capturing immediate changes.
- **Seasonal lags** (48, 336): Use for capturing daily/weekly cycles.
- **Window size:** Should be large enough to smooth noise but small enough to capture relevant patterns. Typical choices are 3–14 for daily, 2–8 for weekly.

---

### **Key Takeaway**

- **Use short lags (1, 2, 3) for recent trends.**
- **Use seasonal lags (48 for daily, 336 for weekly) for recurring patterns.**
- **Choose window sizes based on the amount of history you want to summarize (e.g., 3 for short-term, 7 for a week, 4 for a month).**
- **Always compute statistics up to $t-1$ or further back—never include the current or future value to avoid data leakage.**

---

By thoughtfully selecting lag transformations and window sizes, you provide your forecasting model with rich, relevant historical context—improving both accuracy and interpretability.

# Seasonal Rolling Features: Capturing Repeating Patterns

In time series forecasting, **seasonal rolling features** help models recognize repeating patterns that occur at regular intervals—such as daily or weekly cycles in electricity demand.

## What Are Seasonal Rolling Features?

A **seasonal rolling feature** is a summary statistic (like mean or standard deviation) computed over a moving window, but specifically at a seasonal lag. For example:

- **Daily seasonality:** For half-hourly data, lag 48 corresponds to the same time yesterday.
- **Weekly seasonality:** Lag 336 corresponds to the same time last week.

By applying rolling statistics to these lags, we help the model understand not just what happened recently, but how the current value compares to similar times in the past.

---

## Why Use Seasonal Rolling Features?

- **Capture Recurring Patterns:** Many behaviors (like electricity use) repeat daily or weekly.
- **Smooth Outliers:** Rolling means or stds at seasonal lags help reduce the impact of unusual spikes or drops.
- **Contextualize Current Values:** The model can learn if the current demand is high or low compared to recent days or weeks.

---

## Example: Adding Seasonal Rolling Features with MLForecast

Let's see how to add rolling mean and standard deviation features at daily and weekly lags using the `mlforecast` library:

```python
# We already have an MLForecast instance (mlf) set up with seasonal rolling features
# Let's preprocess the data to generate these features

seasonal_rolling_features = mlf.preprocess(data)
seasonal_rolling_features.head(10)
```

**Explanation:**
- Here, `mlf` is configured to compute rolling means and stds at lag 48 (daily) with window sizes 7 and 14.
- This means, for each time point, we get:
    - The mean and std of the value at the same time over the past 7 and 14 days.

---

## Mathematical Formulation

For a time series $y_t$, the **seasonal rolling mean** at lag $k$ and window size $w$ is:

$$
\text{SeasonalRollingMean}_{t} = \frac{1}{w} \sum_{i=1}^{w} y_{t - i \cdot k}
$$

- $k$ = seasonal lag (e.g., 48 for daily, 336 for weekly)
- $w$ = window size (number of previous seasons to average)

---

## Best Practices

- **Choose lags** that match your data's seasonality (e.g., 48 for daily, 336 for weekly in half-hourly data).
- **Window size** should be large enough to smooth noise but small enough to capture recent trends (e.g., 7 for a week, 14 for two weeks).
- **Combine with other features:** Use alongside short-term lags and rolling features for best results.

---

**Next:**  
We'll explore how these seasonal rolling features improve model performance and interpretability, and visualize their effect on the data!

In [ ]:
from mlforecast.lag_transforms import SeasonalRollingMean, SeasonalRollingStd

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={
        48: [
            SeasonalRollingMean(season_length=48, window_size=3),
            SeasonalRollingStd(season_length=48, window_size=3),
        ]
    },
    freq="30m",
)

mlf.preprocess(data).head(10)

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={
        336: [
            SeasonalRollingMean(season_length=336, window_size=3),
            SeasonalRollingStd(season_length=336, window_size=3),
        ]
    },
    freq="30m",
)

mlf.preprocess(data).head(10)

# Exponentially Weighted Mean (EWM) in Time Series Feature Engineering

## What is the Exponentially Weighted Mean?

The **Exponentially Weighted Mean (EWM)** is a powerful technique for smoothing time series data and capturing recent trends. Unlike a simple rolling mean, which gives equal weight to all values in the window, the EWM assigns **more weight to recent observations** and less to older ones. This makes it especially useful for time series forecasting, where the most recent data often carries the most predictive power.

### Mathematical Definition

For a time series $y_t$, the exponentially weighted mean at time $t$ is defined recursively as:

$$
\text{EWM}_t = \alpha \cdot y_t + (1 - \alpha) \cdot \text{EWM}_{t-1}
$$

- $\alpha$ is the **smoothing factor** (between 0 and 1).
    - **Higher $\alpha$**: More weight to recent values (faster adaptation, less smoothing).
    - **Lower $\alpha$**: More weight to older values (slower adaptation, more smoothing).

---

## Why Use EWM for Feature Engineering?

- **Captures Recent Trends:** Reacts quickly to changes in the data.
- **Smooths Noise:** Reduces the impact of random fluctuations.
- **Flexible Memory:** The "memory" of the mean is controlled by $\alpha$.

---

## Choosing the Right Alpha ($\alpha$)

- **$\alpha$ close to 1** (e.g., 0.8, 0.9):  
  - The EWM reacts quickly to new data.
  - Useful when recent changes are highly predictive.
  - Can be noisy if the data is volatile.

- **$\alpha$ close to 0** (e.g., 0.1, 0.2):  
  - The EWM changes slowly, emphasizing long-term trends.
  - Useful when the series is stable and you want to smooth out short-term noise.

- **Typical values:**  
  - For electricity demand, try $\alpha$ values between 0.1 and 0.5 to start.
  - **Experimentation is key:** Use cross-validation to find the best $\alpha$ for your forecasting model.

---

## Practical Example: Computing EWM with Polars

Let's compute the exponentially weighted mean for our selected household using different $\alpha$ values.


In [ ]:
import math
from mlforecast.lag_transforms import ExponentiallyWeightedMean

## Understanding "Span" as an Alternative to Alpha in Exponentially Weighted Mean (EWM)

When using the **Exponentially Weighted Mean (EWM)** for time series feature engineering, you can control how quickly the mean "forgets" old data by adjusting the **smoothing factor**. In most libraries (including `polars` and `pandas`), you can specify this smoothing either with **alpha** ($\alpha$) or with an alternative parameter called **span**.

---

### What is "Span"?

- **Span** provides a more intuitive way to set the memory of the EWM.
- It represents the "window size" that the EWM is roughly equivalent to, in terms of how much past data it considers.
- **Higher span** means the EWM averages over a longer history (more smoothing).
- **Lower span** means the EWM reacts more quickly to recent changes (less smoothing).

---

### The Relationship Between Alpha and Span

The mathematical relationship between **alpha** ($\alpha$) and **span** is:

$$
\alpha = \frac{2}{\text{span} + 1}
$$

Or, rearranged to solve for span:

$$
\text{span} = \frac{2}{\alpha} - 1
$$

- **Alpha ($\alpha$):** Directly controls the weight given to the most recent observation (range: $0 < \alpha \leq 1$).
- **Span:** Provides an equivalent "window size" for the EWM.

---

### Why Use Span Instead of Alpha?

- **Intuitive Interpretation:** It's often easier to think in terms of "how many periods back" you want the EWM to consider, rather than picking a raw alpha value.
- **Consistency with Rolling Windows:** If you're used to rolling means with a window of, say, 7, you might choose a span of 7 for your EWM to get similar smoothing.

---

### Example Calculation

Suppose you want your EWM to have a similar memory as a 7-period rolling mean:

- Set $\text{span} = 7$
- Compute $\alpha = 2 / (7 + 1) = 0.25$

So, using `span=7` is equivalent to `alpha=0.25`.

---

### Practical Tip

- **Choose the parameter (alpha or span) that makes the most sense for your problem.**
- Most libraries let you specify either one, and will compute the other automatically.

---

### Summary Table

| Parameter | Meaning                              | Formula                        |
|-----------|--------------------------------------|--------------------------------|
| $\alpha$  | Smoothing factor (recent weight)     | $\alpha = \frac{2}{\text{span} + 1}$ |
| span      | Equivalent window size (intuitive)   | $\text{span} = \frac{2}{\alpha} - 1$ |

---

**Key Takeaway:**  
- Use **span** when you want to set the EWM's "memory" in terms of periods, or **alpha** when you want fine control over the weighting.  
- Both control how quickly the EWM adapts to new data, just from different perspectives!

## Choosing a Good "Span" for Exponentially Weighted Mean (EWM) in Half-Hourly Electricity Load Forecasting

When using the **Exponentially Weighted Mean (EWM)** as a feature for half-hourly electricity demand forecasting, the "span" parameter controls how much historical data influences the smoothed value. Let's break down how to choose a good span:

---

### 1. **What Does "Span" Mean?**

- **Span** is an intuitive way to set the "memory" of the EWM.
- A **span of $k$** means the EWM gives similar weight to the most recent $k$ observations as a rolling mean of window $k$.
- **Shorter span:** EWM reacts quickly to recent changes (less smoothing).
- **Longer span:** EWM smooths over more history (more smoothing).

---

### 2. **Domain Knowledge: Typical Patterns in Electricity Demand**

- **Daily cycles:** For half-hourly data, there are 48 intervals per day.
- **Weekly cycles:** 336 intervals per week ($48 \times 7$).
- **Short-term trends:** Demand can change rapidly due to weather, time of day, or special events.

---

### 3. **Recommended Span Values**

- **Short-term smoothing:**  
    - **Span = 6 to 12** (3–6 hours): Captures very recent trends, useful for highly volatile demand.
- **Daily smoothing:**  
    - **Span = 24 to 48** (12–24 hours, or 1 day): Smooths over a full day, capturing daily cycles.
- **Weekly smoothing:**  
    - **Span = 168 to 336** (3.5–7 days): Smooths over several days or a week, capturing weekly seasonality.

**Typical starting points:**
- **Span = 24** (12 hours): Good for capturing half-day trends.
- **Span = 48** (1 day): Good for daily smoothing.
- **Span = 336** (1 week): Good for weekly smoothing.

---

### 4. **How to Choose the Best Span?**

- **Experiment:** Try several spans (e.g., 12, 24, 48, 168, 336) and evaluate model performance.
- **Cross-validation:** Use out-of-sample testing to see which span helps your model forecast best.
- **Combine features:** It's common to include multiple EWM features with different spans to capture both short-term and long-term trends.

---

### 5. **Summary Table**

| Span | Meaning                | Typical Use                |
|------|------------------------|----------------------------|
| 6    | 3 hours                | Very short-term smoothing  |
| 24   | 12 hours               | Half-day trend             |
| 48   | 1 day                  | Daily smoothing            |
| 168  | 3.5 days               | Multi-day smoothing        |
| 336  | 1 week                 | Weekly smoothing           |

---

### **Key Takeaway**

> For half-hourly electricity load forecasting, start with **span values of 24, 48, and 336** to capture half-day, daily, and weekly patterns.  
> Use cross-validation to fine-tune these values for your specific dataset and forecasting goal.

---

**Next:**  
Let's see how to compute EWM features with different spans in code!

In [ ]:
t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
for alpha in [0.3, 0.5, 0.8]:
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    span = (2 - alpha) / alpha
    halflife = math.log(1 - alpha) / math.log(0.5)
    plot_df[f"Alpha={alpha} | Span={span:.2f}"] = weights

fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
)
fig.update_layout(
    autosize=False,
    width=1200,
    height=500,
)
fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
t = np.arange(25).tolist()
plot_df = pd.DataFrame({"Timesteps behind t": t})
# Instead of specifying alpha, let's specify span values (24, 48, 336) and compute the corresponding alpha for each.
for span in [3, 6, 12, 24, 48, 336]:
    # The relationship between alpha and span is: alpha = 2 / (span + 1)
    alpha = 2 / (span + 1)
    weights = [alpha * math.pow((1 - alpha), i) for i in t]
    plot_df[f"Span={span} | Alpha={alpha:.3f}"] = weights

fig = px.line(
    pd.melt(plot_df, id_vars="Timesteps behind t", var_name="Parameters"),
    x="Timesteps behind t",
    y="value",
    facet_col="Parameters",
    facet_col_wrap=3,
)
fig.update_layout(
    autosize=False,
    width=1200,
    height=500,
)
fig.update_annotations(font=dict(size=16))
fig.show()

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    lag_transforms={1: [ExponentiallyWeightedMean(alpha=0.25)]},
    freq="30m",
)

mlf.preprocess(data).head(10)

# Temporal Embedding Features in Time Series Forecasting

In addition to lag and rolling features, **temporal embedding features** help your model understand the position of each observation in time. These features encode information about the calendar, periodic cycles, and the passage of time, making them essential for capturing seasonality and trends.

---

## 1. Calendar Features

**Calendar features** extract information from the timestamp, such as:

- **Hour of day** ($0$–$23$)
- **Day of week** ($0$–$6$, where $0$ is Monday)
- **Day of month** ($1$–$31$)
- **Month** ($1$–$12$)
- **Is weekend** (binary: $1$ if Saturday/Sunday, $0$ otherwise)
- **Is holiday** (if holiday data is available)

**Why use them?**
- Electricity demand often varies by hour, day, or month due to human routines and holidays.
- These features help the model learn patterns like "demand is higher on weekdays at 6pm" or "lower on weekends."

---

## 2. Fourier Terms

**Fourier terms** are mathematical features that help capture smooth, repeating seasonal patterns (like daily or yearly cycles) using sine and cosine functions.

For a given period $P$ (e.g., $P=48$ for daily seasonality in half-hourly data), the $k$-th Fourier term at time $t$ is:

$$
\text{Fourier}_k(t) = \sin\left(\frac{2\pi k t}{P}\right), \quad \cos\left(\frac{2\pi k t}{P}\right)
$$

- **Why use them?**  
    - They allow models to fit smooth, cyclical patterns that calendar features alone can't capture.
    - Especially useful for capturing complex seasonality (e.g., annual cycles, multiple seasonalities).

---

## 3. Time Elapsed Features

**Time elapsed** features encode the passage of time, such as:

- **Time since start** (e.g., number of periods since the beginning)
- **Time since last event** (e.g., time since last holiday or anomaly)
- **Cumulative time** (useful for trend detection)

**Why use them?**
- They help the model detect and learn long-term trends or gradual changes over time.

---

## Practical Example: Adding Temporal Embedding Features with Polars

Let's see how to create these features for our electricity demand data using `polars`:

## Key Takeaways

- **Calendar features** help capture effects tied to the clock and calendar.
- **Fourier terms** are powerful for modeling smooth, repeating cycles.
- **Time elapsed** features help detect trends and long-term changes.
- Combining these temporal embeddings with lag and rolling features gives your model a rich understanding of both short-term and long-term temporal patterns.

---

**Next:**  
We'll visualize some of these features and discuss how they improve model interpretability and forecasting accuracy!

In [ ]:
calendar_features = [
    "month",
    "quarter",
    "week",
    "day",
    "weekday",
    "hour",
    "minute",
]

In [ ]:
# Set up MLForecast with lags only (no additional features for now)
mlf = MLForecast(
    models=[],
    freq="30m",
    date_features=calendar_features,
)

mlf.preprocess(data).head(10)

In [ ]:
from utilsforecast.feature_engineering import fourier, pipeline
from functools import partial

In [ ]:
features = [
    partial(
        fourier, season_length=2 * 24, k=10
    ),  # Daily seasonality (48 observations per day)
    partial(
        fourier, season_length=2 * 24 * 7, k=5
    ),  # Weekly seasonality (336 observations per week)
    partial(
        fourier, season_length=2 * 24 * 365, k=3
    ),  # Annual seasonality (approx. 17520 observations per year)
]
data_fourier, data_futr_fourier = pipeline(
    data,
    features=features,
    freq="30m",
    h=48,  # Horizon for future features
)
data_fourier.head(10)

In [ ]:
data.with_columns(pl.col("ds").cast(pl.Int64).truediv(10e9).alias("ds_int"))